<a href="https://colab.research.google.com/github/the-craft-labs/training_bactrocera/blob/main/training_bactrocera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bactrocera — chia dataset & train YOLO

Notebook cho dữ liệu YOLO **chưa chia tập** và **tên file không theo quy cách**.

Nhóm ảnh được suy ra từ EXIF timestamp, thư mục con và perceptual hash thay vì từ tên file,
để ảnh gần trùng nhau (cùng một tấm dính chụp nhiều lần) không bị tách sang hai tập khác nhau —
đó là nguồn leakage làm mAP ảo cao.

Chạy tuần tự từ trên xuống. Sau mỗi bước có phần **cách đọc kết quả** — đọc trước khi chạy tiếp.

Cấu trúc đầu vào mong đợi (thư mục con lồng nhau bao nhiêu cấp cũng được):

```
dataset/
├── images/   *.jpg
└── labels/   *.txt
```

## 0. Cấu hình

In [ ]:
# !pip install -q ultralytics pillow numpy matplotlib pyyaml

from pathlib import Path
from datetime import datetime
from collections import Counter
from PIL import Image
import random, shutil, csv, json, yaml
import numpy as np

# ── đường dẫn ──
SRC     = Path("dataset")            # chứa images/ và labels/
DST     = Path("dataset_split")
CLASSES = ["fly"]                    # đúng thứ tự class id trong file .txt
SPLIT   = (0.70, 0.20, 0.10)         # train / val / test
SEED    = 42

# ── gom nhóm chống leakage ──
USE_TIME        = True    # dùng EXIF/mtime để cụm theo phiên chụp
USE_FOLDER      = False   # True nếu thư mục con tương ứng bẫy / đợt chụp
SESSION_GAP_MIN = 120     # cách nhau > 2h coi như phiên chụp khác
DHASH_MAX_DIST  = 12      # /64 bit — càng nhỏ càng chặt

# ── train ──
MODEL   = "yolo11s.pt"    # yolo11n nếu cần nhẹ; yolo11s-p2.yaml nếu ruồi rất nhỏ
IMGSZ   = 960
EPOCHS  = 300
BATCH   = 8
PATIENCE = 50             # early stopping
DEVICE  = 0               # "cpu" nếu không có GPU

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
random.seed(SEED); np.random.seed(SEED)
print("SRC:", SRC.resolve())

## 1. Quét file và kiểm kê metadata

In [ ]:
imgs = sorted(p for p in (SRC / "images").rglob("*") if p.suffix.lower() in IMG_EXT)
assert imgs, f"Không tìm thấy ảnh trong {SRC/'images'}"

def exif_dt(p):
    # DateTimeOriginal (36867) > DateTimeDigitized (36868) > DateTime của IFD0 (306)
    try:
        ex  = Image.open(p).getexif()
        try:
            sub = ex.get_ifd(0x8769)
        except Exception:
            sub = {}
        raw = sub.get(36867) or sub.get(36868) or ex.get(306)
        if raw:
            return datetime.strptime(str(raw).strip(), "%Y:%m:%d %H:%M:%S")
    except Exception:
        pass
    return None

recs = []
for p in imgs:
    dt, src = exif_dt(p), "exif"
    if dt is None:
        dt, src = datetime.fromtimestamp(p.stat().st_mtime), "mtime"
    with Image.open(p) as im:
        wh = im.size
    recs.append({"path": p, "dt": dt, "src": src, "wh": wh,
                 "folder": p.parent.relative_to(SRC / "images").as_posix() or "."})

n = len(recs)
print("tổng ảnh        :", n)
print("nguồn thời gian :", dict(Counter(r["src"] for r in recs)))
print("khoảng ngày     :", min(r["dt"] for r in recs).date(), "→", max(r["dt"] for r in recs).date())
print("số ngày riêng   :", len({r["dt"].date() for r in recs}))
print("độ phân giải    :", Counter(r["wh"] for r in recs).most_common(5))
print("thư mục con     :", Counter(r["folder"] for r in recs).most_common(10))

### Cách đọc kết quả

| Thấy gì | Nghĩa là | Làm gì |
|---|---|---|
| `nguồn thời gian` phần lớn là `mtime` | EXIF đã bị strip (thường do gửi qua Zalo/Messenger); `mtime` chỉ là ngày copy file, vô nghĩa | đặt `USE_TIME = False`, chỉ dựa vào hash |
| `thư mục con` chia thành các nhóm rõ ràng | rất có thể tương ứng bẫy hoặc đợt chụp — tín hiệu đáng tin hơn hash | đặt `USE_FOLDER = True` |
| `số ngày riêng` xấp xỉ số ảnh | mỗi ảnh một ngày, ít khả năng có burst frame | leakage chủ yếu đến từ tấm dính lặp lại, để hash lo |
| `độ phân giải` nhiều loại | dữ liệu trộn từ nhiều máy | ghi nhận lại, sau này đối chiếu với ảnh OV5640 |

Sửa cấu hình ở cell 0 rồi chạy lại cell trên nếu cần.

## 2. Gom nhóm ảnh liên quan

In [ ]:
def dhash(p, size=8):
    with Image.open(p) as im:
        a = np.asarray(im.convert("L").resize((size + 1, size)), dtype=np.int16)
    return np.packbits((a[:, 1:] > a[:, :-1]).ravel())

parent = list(range(n))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra

# (1) cùng phiên chụp
if USE_TIME:
    order = sorted(range(n), key=lambda i: recs[i]["dt"])
    for a, b in zip(order, order[1:]):
        if (recs[b]["dt"] - recs[a]["dt"]).total_seconds() <= SESSION_GAP_MIN * 60:
            union(a, b)

# (2) cùng thư mục
if USE_FOLDER:
    by_folder = {}
    for i, r in enumerate(recs):
        by_folder.setdefault(r["folder"], []).append(i)
    for ids in by_folder.values():
        for j in ids[1:]:
            union(ids[0], j)

# (3) ảnh gần trùng
H    = np.stack([dhash(r["path"]) for r in recs])
POPC = np.unpackbits(np.arange(256, dtype=np.uint8)[:, None], axis=1).sum(1).astype(np.uint8)
D    = np.zeros((n, n), np.uint8)
for i in range(0, n, 512):
    D[i:i+512] = POPC[H[i:i+512, None, :] ^ H[None, :, :]].sum(-1)

iu = np.triu_indices(n, 1)
hist = np.bincount(D[iu] // 8 * 8, minlength=64)
print("phân bố khoảng cách dHash (gộp bin 8):")
for b in range(0, 64, 8):
    print(f"  {b:2d}-{b+7:2d} bit : {hist[b]:>9,}")

for a, b in zip(*np.where(np.triu(D, 1) <= DHASH_MAX_DIST)):
    union(int(a), int(b))

groups = {}
for i in range(n):
    groups.setdefault(find(i), []).append(recs[i]["path"])

sizes = sorted((len(v) for v in groups.values()), reverse=True)
print(f"\n{len(groups)} nhóm / {n} ảnh")
print("kích thước nhóm lớn nhất :", sizes[:10])
print("số nhóm chỉ có 1 ảnh     :", sum(1 for s in sizes if s == 1))
print("nhóm lớn nhất chiếm      : %.1f%% dữ liệu" % (100 * sizes[0] / n))

### Hiệu chỉnh `DHASH_MAX_DIST`

- **Số nhóm ≈ số ảnh** → không có ảnh nào gần trùng, chia ngẫu nhiên vốn đã an toàn.
- **Số nhóm tụt còn vài chục** trong khi bạn biết có hàng trăm tấm dính khác nhau → ngưỡng quá lỏng, hạ về `8` rồi `6`.
- **Nhóm lớn nhất chiếm > 30% dữ liệu** → mọi thứ bị nối thành một cụm khổng lồ (thường do `USE_TIME` gộp cả buổi chụp). Giảm `SESSION_GAP_MIN` hoặc tắt `USE_TIME`.

Nhìn thêm phân bố dHash: nếu có khe rõ giữa bin `0-7` và các bin sau, đặt ngưỡng ngay tại khe đó.

## 3. Chia tập và copy kèm đổi tên

In [ ]:
def find_label(img):
    rel = img.parent.relative_to(SRC / "images")
    for c in (SRC / "labels" / f"{img.stem}.txt",
              SRC / "labels" / rel / f"{img.stem}.txt",
              img.with_suffix(".txt")):
        if c.exists():
            return c
    return None

# chia tham lam: nhóm lớn xếp trước, luôn bỏ vào tập đang thiếu nhất
keys = list(groups)
random.Random(SEED).shuffle(keys)
keys.sort(key=lambda k: -len(groups[k]))

target  = {s: n * f for s, f in zip(("train", "val", "test"), SPLIT)}
buckets = {s: [] for s in target}
filled  = {s: 0 for s in target}
for k in keys:
    s = max(target, key=lambda x: target[x] - filled[x])
    buckets[s].append(k); filled[s] += len(groups[k])

if DST.exists():
    shutil.rmtree(DST)

mapping, rows = [], []
for split, ks in buckets.items():
    (DST / "images" / split).mkdir(parents=True, exist_ok=True)
    (DST / "labels" / split).mkdir(parents=True, exist_ok=True)
    n_img = n_box = n_bg = 0
    for gi, k in enumerate(ks):
        for ii, img in enumerate(sorted(groups[k])):
            new = f"{split[:2]}_{gi:04d}_{ii:03d}"
            shutil.copy2(img, DST / "images" / split / f"{new}{img.suffix.lower()}")
            lbl = find_label(img)
            out = DST / "labels" / split / f"{new}.txt"
            if lbl:
                shutil.copy2(lbl, out)
                n_box += sum(1 for l in out.read_text().splitlines() if l.strip())
            else:
                out.write_text("")          # ảnh nền — YOLO coi là negative sample
                n_bg += 1
            n_img += 1
            mapping.append([split, gi, f"{new}{img.suffix.lower()}", str(img)])
    rows.append((split, len(ks), n_img, n_box, n_bg))

with open(DST / "mapping.csv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows([["split", "group", "new_name", "original_path"]] + mapping)

print(f"{'split':<7}{'groups':>8}{'images':>8}{'%':>7}{'boxes':>9}{'no_label':>10}")
for s, g, i, b, bg in rows:
    print(f"{s:<7}{g:>8}{i:>8}{100*i/n:>6.1f}%{b:>9}{bg:>10}")
print("\nmapping.csv →", (DST / 'mapping.csv').resolve())

### Kiểm hai thứ ở bảng trên

1. **Cột `%`** có bám sát 70/20/10 không. Lệch nhiều nghĩa là một nhóm khổng lồ đã nuốt cả tập — quay lại bước 2 siết ngưỡng.
2. **Cột `no_label`** — nếu bằng đúng số ảnh thì `find_label()` không tìm ra file `.txt`. Kiểm tra lại cấu trúc thư mục `labels/` và sửa hàm trước khi train.

`mapping.csv` giữ liên kết tên mới ↔ đường dẫn gốc, ghép thẳng vào `manifest.json` cho phần nghiệm thu.

## 4. Validate label và đo kích thước vật thể

In [ ]:
bad, areas = [], []
for split in ("train", "val", "test"):
    for lp in sorted((DST / "labels" / split).glob("*.txt")):
        for ln, line in enumerate(lp.read_text().splitlines(), 1):
            t = line.split()
            if not t:
                continue
            if len(t) != 5:
                bad.append((split, lp.name, ln, f"{len(t)} cột thay vì 5")); continue
            try:
                cid = int(float(t[0])); x, y, w, h = map(float, t[1:])
            except ValueError:
                bad.append((split, lp.name, ln, "không parse được số")); continue
            if not 0 <= cid < len(CLASSES):
                bad.append((split, lp.name, ln, f"class id {cid} ngoài phạm vi"))
            if not all(0.0 <= v <= 1.0 for v in (x, y, w, h)):
                bad.append((split, lp.name, ln, "toạ độ ngoài [0,1] — có thể chưa normalize"))
            if w <= 0 or h <= 0:
                bad.append((split, lp.name, ln, "box rỗng"))
            else:
                areas.append((w * h) ** 0.5 * IMGSZ)     # cạnh tương đương, px @ IMGSZ

print(f"{len(bad)} dòng lỗi")
for r in bad[:20]:
    print("  ", r)
if len(bad) > 20:
    print(f"   ... còn {len(bad)-20} dòng")

if areas:
    a = np.array(areas)
    q = np.percentile(a, [5, 25, 50, 75, 95])
    print(f"\n{len(a)} box | cạnh tương đương @ imgsz={IMGSZ} (px)")
    print("  p5=%.1f  p25=%.1f  median=%.1f  p75=%.1f  p95=%.1f" % tuple(q))
    print("  box < 16px: %.1f%%   |   box < 32px: %.1f%%" % (100*(a<16).mean(), 100*(a<32).mean()))

### Chọn `IMGSZ` và kiến trúc theo median

| Median cạnh box | Hành động |
|---|---|
| > 32 px | `yolo11s.pt`, `IMGSZ=960` là đủ |
| 16 – 32 px | nâng `IMGSZ` lên 1280 |
| < 16 px | đổi `MODEL = "yolo11s-p2.yaml"` — thêm detection head ở stride 4, chậm hơn nhưng bắt vật thể nhỏ tốt hơn hẳn |

Nếu `% box < 16px` cao, đừng cố bù bằng epoch nhiều hơn; stride của model mới là thứ chặn trần độ chính xác.

## 5. Xem thử ảnh kèm box

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

pool   = sorted((DST / "images" / "train").glob("*"))
sample = random.Random(SEED).sample(pool, k=min(6, len(pool)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, p in zip(axes.ravel(), sample):
    with Image.open(p) as im:
        W, H = im.size
        ax.imshow(im)
    lp = DST / "labels" / "train" / f"{p.stem}.txt"
    k = 0
    if lp.exists():
        for line in lp.read_text().splitlines():
            t = line.split()
            if len(t) != 5:
                continue
            _, x, y, w, h = (float(v) for v in t)
            ax.add_patch(mpatches.Rectangle(((x-w/2)*W, (y-h/2)*H), w*W, h*H,
                                            fill=False, lw=1.2, edgecolor="lime"))
            k += 1
    ax.set_title(f"{p.name} — {k} box", fontsize=9)
    ax.axis("off")
for ax in axes.ravel()[len(sample):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

Box phải ôm đúng con ruồi. Nếu box lệch hệ thống hoặc lật trục, file `.txt` đang ở format khác
(Pascal VOC pixel, hoặc `x1 y1 x2 y2`) chứ không phải YOLO normalized — phải convert trước khi train.

## 6. Sinh data.yaml

In [ ]:
data_yaml = DST / "data.yaml"
data_yaml.write_text(yaml.safe_dump({
    "path":  str(DST.resolve()),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "names": {i: c for i, c in enumerate(CLASSES)},
}, sort_keys=False, allow_unicode=True), encoding="utf-8")

print(data_yaml.read_text(encoding="utf-8"))

## 7. Train — early stopping + cosine LR + EMA

Ba cơ chế tương ứng:

- **Early stopping** — `patience`: dừng nếu *fitness* không cải thiện sau N epoch.
- **LR scheduling** — `cos_lr` + `lr0` + `lrf`: cosine decay từ `lr0` xuống `lr0 * lrf`, có warmup ở đầu.
- **EMA** — bật mặc định trong Ultralytics, không có flag tắt. Đây mới là phần *tự cải thiện* thật sự,
  chứ không chỉ chọn checkpoint tốt nhất.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)

results = model.train(
    data=str(data_yaml),
    project="runs/bactrocera", name="v1", seed=SEED,

    # ── vòng train ──
    epochs=EPOCHS,
    patience=PATIENCE,      # EARLY STOPPING
    batch=BATCH,
    imgsz=IMGSZ,
    device=DEVICE,

    # ── LR SCHEDULING ──
    optimizer="AdamW",
    lr0=0.001,              # LR khởi điểm
    lrf=0.01,               # LR cuối = lr0 * lrf = 1e-5
    cos_lr=True,            # cosine decay thay vì linear
    warmup_epochs=3.0,
    warmup_bias_lr=0.1,
    weight_decay=0.0005,

    # ── augmentation ──
    degrees=180,            # ảnh chụp từ trên xuống, không có hướng chuẩn
    flipud=0.5, fliplr=0.5,
    scale=0.5, translate=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,   # bù chênh lệch màu phone vs OV5640
    mosaic=1.0,
    close_mosaic=15,        # tắt mosaic 15 epoch cuối để hội tụ sạch

    plots=True,
)

RUN  = Path(results.save_dir)
BEST = RUN / "weights" / "best.pt"
LAST = RUN / "weights" / "last.pt"
print("\nrun dir:", RUN.resolve())

## 8. Đọc checkpoint dict

In [ ]:
import torch

ckpt = torch.load(BEST, map_location="cpu", weights_only=False)
print("keys:", list(ckpt.keys()))
for k in ("epoch", "best_fitness", "date", "version"):
    print(f"  {k:<13}: {ckpt.get(k)}")
print("\nkích thước file:")
for f in (BEST, LAST):
    print(f"  {f.name:<9} {f.stat().st_size/1e6:6.1f} MB")

### Vì sao `best_fitness` và `optimizer` lại ra `None`

File `.pt` là một dict Python. **Trong lúc train** nó gồm `model`, `ema`, `updates`, `optimizer`,
`epoch`, `best_fitness`, `train_args`, `train_metrics`.

Nhưng khi train kết thúc, Ultralytics gọi `strip_optimizer()` lên `best.pt` và `last.pt`:
hàm này **ghi đè `model` bằng `ema`**, rồi set `optimizer` / `ema` / `updates` / `best_fitness` về `None`
và `epoch = -1`. Nên `best.pt` bạn cầm về **đã chính là EMA weights** — nhẹ hơn khoảng một nửa và
thường tốt hơn weights gốc. Muốn giữ dict đầy đủ để resume thì phải copy checkpoint ra chỗ khác
*trong lúc* đang train.

`best.pt` được chọn theo **fitness** `= 0.1·mAP50 + 0.9·mAP50-95`, không phải theo loss.

## 9. Đánh giá trên tập test

In [ ]:
m   = YOLO(BEST)
res = m.val(data=str(data_yaml), split="test", imgsz=IMGSZ,
            conf=0.001, iou=0.6, max_det=1000)

print(f"mAP50-95 : {res.box.map:.4f}")
print(f"mAP50    : {res.box.map50:.4f}")
print(f"Precision: {res.box.mp:.4f}")
print(f"Recall   : {res.box.mr:.4f}")
print(f"fitness  : {0.1*res.box.map50 + 0.9*res.box.map:.4f}")

## 10. Dò ngưỡng `conf` theo sai số đếm

Bài toán ở đây là **đếm** chứ không phải phát hiện, nên mAP không phải chỉ số cuối cùng.
Cái cần tối ưu là MAE giữa số box dự đoán và số box thật trên mỗi ảnh.

Ngưỡng `conf` tối ưu cho đếm thường **thấp hơn** ngưỡng tối ưu cho F1: false negative
(bỏ sót ruồi) làm lệch đường cong mật độ quần thể nặng hơn false positive, vì sai số bỏ sót
tích lũy theo một hướng qua các ngày còn false positive thì phân tán ngẫu nhiên.

In [ ]:
test_imgs = sorted((DST / "images" / "test").glob("*"))
gt = np.array([
    sum(1 for l in (DST / "labels" / "test" / f"{p.stem}.txt").read_text().splitlines() if l.strip())
    for p in test_imgs
])

print(f"{'conf':>6}{'MAE':>9}{'bias':>9}{'MAPE':>9}")
best_conf, best_mae = None, float("inf")
for c in np.arange(0.05, 0.65, 0.05):
    preds = m.predict(test_imgs, conf=float(c), iou=0.5, imgsz=IMGSZ,
                      max_det=1000, verbose=False, stream=True)
    pc   = np.array([len(r.boxes) for r in preds])
    mae  = np.abs(pc - gt).mean()
    bias = (pc - gt).mean()
    mape = 100 * (np.abs(pc - gt) / np.maximum(gt, 1)).mean()
    print(f"{c:>6.2f}{mae:>9.2f}{bias:>+9.2f}{mape:>8.1f}%")
    if mae < best_mae:
        best_mae, best_conf = mae, float(c)

print(f"\nconf tối ưu cho đếm: {best_conf:.2f}  (MAE={best_mae:.2f})")
print("→ dùng giá trị này cho inference trên backend, không dùng mặc định 0.25")

## Bước tiếp theo

- Ghi `best_conf`, `mAP50-95` và hash của `mapping.csv` vào bảng `model_runs` để lần retrain sau đối chiếu được.
- Trước khi triển khai: val lại trên một tập nhỏ ảnh chụp bằng **OV5640** thật. Chênh lệch mAP giữa
  ảnh phone và ảnh OV5640 chính là chỉ số đo domain gap — nếu tụt nhiều thì cần fine-tune trên ảnh camera thật
  chứ không phải augment thêm.
- Export cho inference: `m.export(format="onnx", imgsz=IMGSZ, simplify=True)`.